In [1]:
import scipy.io as sio
import os

# Carpeta donde están los archivos .mat
folder = "stroke-rehab"

# Lista de archivos .mat dentro de la carpeta
mat_files = [f for f in os.listdir(folder) if f.endswith(".mat")]

# Diccionario donde guardaremos los datos cargados
data = {}

for filename in mat_files:
    filepath = os.path.join(folder, filename)
    data[filename] = sio.loadmat(filepath)

print("Carga completada.")


Carga completada.


In [2]:
import numpy as np
from sklearn.model_selection import train_test_split

def prepare_eeg_dataset_3class(signals, triggers, fs=256):
    triggers = triggers.flatten()

    # Duraciones
    trigger_duration = 8 * fs

    rest_duration = 2 * fs
    task_duration = 6 * fs

    window_size = int(2 * fs)

    X, y = [], []

    i = 0
    n = len(triggers)

    while i < n:
        if triggers[i] in [-1, 1]:
            current_trigger = triggers[i]

            start = i
            while i < n and triggers[i] == current_trigger:
                i += 1
            end = i

            if (end - start) >= trigger_duration:

                # ----------- REPOSO (0–2 s) -----------
                rest_start = start
                rest_end = rest_start + rest_duration

                rest_signal = signals[rest_start:rest_end]

                for w in range(0, len(rest_signal) - window_size + 1, window_size):
                    window = rest_signal[w:w+window_size]
                    X.append(window)
                    y.append(2)  # clase reposo

                # ----------- TAREA (2–8 s) -----------
                task_start = start + rest_duration
                task_end = task_start + task_duration

                if task_end <= len(signals):
                    task_signal = signals[task_start:task_end]

                    label = 0 if current_trigger == -1 else 1

                    for w in range(0, len(task_signal) - window_size + 1, window_size):
                        window = task_signal[w:w+window_size]
                        X.append(window)
                        y.append(label)

        else:
            i += 1

    return np.array(X), np.array(y)

In [3]:
for k in data.keys():
    signals = data[k]['y']
    triggers = data[k]['trig']

    X, y = prepare_eeg_dataset_3class(signals, triggers)
    np.savez(f'datos_procesados_triple/{k}.npz', X=X, y=y)

    print(k)
    print('-'*30)
    print("X shape:", X.shape)
    print("y shape:", y.shape)
    print("Distribución etiquetas:", np.bincount(y))
    print()


P1_post_test.mat
------------------------------
X shape: (320, 512, 16)
y shape: (320,)
Distribución etiquetas: [120 120  80]

P1_post_training.mat
------------------------------
X shape: (320, 512, 16)
y shape: (320,)
Distribución etiquetas: [120 120  80]

P1_pre_test.mat
------------------------------
X shape: (320, 512, 16)
y shape: (320,)
Distribución etiquetas: [120 120  80]

P1_pre_training.mat
------------------------------
X shape: (320, 512, 16)
y shape: (320,)
Distribución etiquetas: [120 120  80]

P2_post_test.mat
------------------------------
X shape: (320, 512, 16)
y shape: (320,)
Distribución etiquetas: [120 120  80]

P2_post_training.mat
------------------------------
X shape: (320, 512, 16)
y shape: (320,)
Distribución etiquetas: [120 120  80]

P2_pre_test.mat
------------------------------
X shape: (320, 512, 16)
y shape: (320,)
Distribución etiquetas: [120 120  80]

P2_pre_training.mat
------------------------------
X shape: (320, 512, 16)
y shape: (320,)
Distribució